Data ingestion pipeline

In [1]:
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


In [ ]:
!pip install -U chromadb pypdf langchain-text-splitters

In [2]:
data_path = "C:/Users/User/OneDrive/1_AI/0_data/ocbc_risk_kb/"

In [3]:
from pathlib import Path
from datetime import date
import chromadb
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [4]:
data_path = Path("C:/Users/User/OneDrive/1_AI/0_data/ocbc_risk_kb")
chroma_path = "./my_chroma_db"

In [5]:
client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_or_create_collection(name="ocbc_risk_kb")

document_metadata = {
    # "example.pdf": {
    #     "issue_date": "2025-01-15",
    #     "hyperlink": "https://example.com/example.pdf",
    # }
}

documents = []
metadatas = []
ids = []

for pdf_path in sorted(data_path.rglob("*.pdf")):
    reader = PdfReader(str(pdf_path))
    doc_name = pdf_path.name

    custom_metadata = document_metadata.get(
        doc_name,
        {
            "issue_date": "",
            "hyperlink": "",
        },
    )

    for page_num, page in enumerate(reader.pages, start=1):
        page_text = (page.extract_text() or "").strip()

        if not page_text:
            continue

        chunk_id = f"{pdf_path.stem}-page-{page_num}"

        documents.append(page_text)
        ids.append(chunk_id)
        metadatas.append(
            {
                "chunk_id": chunk_id,
                "doc_name": doc_name,
                "page_num": page_num,
                "issue_date": custom_metadata["issue_date"],
                "hyperlink": custom_metadata["hyperlink"],
            }
        )

batch_size = 100

for start in range(0, len(documents), batch_size):
    end = start + batch_size

    collection.upsert(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end],
    )

print(f"Ingested {len(documents)} PDF pages.")

Overwriting cache for 0 952
C:\Users\User\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:16<00:00, 5.17MiB/s]


Ingested 19 PDF pages.


In [6]:
sample = collection.get(
    limit=1,
    include=["documents", "metadatas"],
)

if sample["ids"]:
    print("ID:", sample["ids"][0])
    print("\nMetadata:")
    for key, value in sample["metadatas"][0].items():
        print(f"{key}: {value}")

    print("\nPage text:")
    print(sample["documents"][0])
else:
    print("The collection is empty.")

ID: ocbc-2011-risk-management-page-1

Metadata:
chunk_id: ocbc-2011-risk-management-page-1
hyperlink: 
page_num: 1
issue_date: 
doc_name: ocbc-2011-risk-management.pdf

Page text:
Risk management
(This section forms an integral part of OCBC’s audited financial statements)
DEVELOPmENTS IN 2011 
During the year, OCBC Group remained focused on our key clients 
and markets in Asia. This strategy provided us with healthy and 
strong broad based growth, including increased contribution from 
our wealth management business through Bank of Singapore 
(“BOS”). Loan origination was predominantly focused on firms 
with strong risk ratings as we carefully increased our activities 
within internal risk limits. Our franchise in China further grew 
with emphasis on secured lending and loans to selected top tier 
corporates and financial institutions. We also continued to integrate 
and consolidate our wealth management platform into BOS and 
align its risk approaches and practices towards Group stand

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()

result = client.models.embed_content(
    model="gemini-embedding-001", # text only; gemini-embedding-2 multimodal
    contents="What is the meaning of life?",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        output_dimensionality=768, # Common dimensions are 768, 1536, or 3072. The default is typically 3072.
    ),
)

embedding = result.embeddings[0].values

print("Vector dimension:", len(embedding))

Vector dimension: 768
